# LAB 2 — BGE-Reranker: Reordenando Resultados com Cross-Encoder
## MBA RAG & CAG Aplicados a Direito e Segurança Pública — Aula 3

**Objetivo:** Aplicar o BGE-Reranker-v2-m3 após o retrieval do OpenSearch e comparar
os top-5 resultados antes e depois do reranking.

**Entregáveis:**
- Pipeline de reranking integrado ao OpenSearch
- Tabela comparativa: top-5 antes vs depois para 5 queries
- Análise: qual query teve maior mudança no ranking?

**Tempo estimado:** 40 minutos  
**GPU recomendada:** T4 ou superior (Colab)


## ⚙️ Passo 1 — Instalação

In [ ]:
!pip install transformers torch sentence-transformers opensearch-py pandas -q
print("✅ Instalação concluída")

In [ ]:
import os, json, time
import torch
import pandas as pd
import numpy as np
from typing import List, Dict

OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST", "localhost")
OPENSEARCH_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
INDEX_NAME      = os.getenv("INDEX_NAME", "corpus_juridico")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device disponível: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 🔌 Passo 2 — Conectar OpenSearch

In [ ]:
from opensearchpy import OpenSearch

os_client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=("admin", os.getenv("OPENSEARCH_PASSWORD", "admin")),
    use_ssl=False, verify_certs=False
)

try:
    count = os_client.count(index=INDEX_NAME)["count"]
    print(f"✅ OpenSearch conectado! Documentos no índice: {count}")
    OPENSEARCH_OK = True
except Exception as e:
    print(f"⚠️  OpenSearch não disponível: {e}")
    print("   Lab executará em modo simulado")
    OPENSEARCH_OK = False

## 🤖 Passo 3 — Carregar BGE-Reranker

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
print(f"⏳ Carregando {RERANKER_MODEL}...")
print("   (~560MB — pode demorar na 1ª execução)")

try:
    reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL)
    reranker_model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    )
    reranker_model.eval().to(device)
    RERANKER_OK = True
    print(f"✅ BGE-Reranker carregado no device: {device}")
    print(f"   Parâmetros: {sum(p.numel() for p in reranker_model.parameters()):,}")
except Exception as e:
    print(f"⚠️  Reranker não disponível: {e}")
    print("   Lab executará com scores simulados")
    RERANKER_OK = False

## 🔎 Passo 4 — Função de Retrieval

In [ ]:
def retrieval_opensearch(query: str, top_k: int = 20) -> List[Dict]:
    """Busca híbrida (texto + metadados) no OpenSearch.
    
    Retorna top_k documentos ordenados por score BM25/TF-IDF.
    Em modo simulado, usa corpus hardcoded do dataset.
    """
    if not OPENSEARCH_OK:
        # Modo simulado — usar docs do dataset local
        try:
            with open("../datasets/corpus_juridico_aula3.json", encoding="utf-8") as f:
                dataset = json.load(f)
            docs = dataset["documentos"]
            # Simular ranking com ordem fixa
            return [
                {
                    "id": d["id"],
                    "score_bm25": round(0.95 - i * 0.03, 3),
                    "texto": d["texto"],
                    "fonte": d.get("numero", d["id"]),
                    "tipo": d.get("tipo", "desconhecido")
                }
                for i, d in enumerate(docs[:top_k])
            ]
        except Exception:
            return []
    
    response = os_client.search(
        index=INDEX_NAME,
        body={
            "query": {
                "multi_match": {
                    "query": query,
                    "fields": ["texto^2", "ementa^1.5", "palavras_chave"],
                    "type": "best_fields"
                }
            },
            "size": top_k
        }
    )
    
    return [
        {
            "id": h["_id"],
            "score_bm25": round(h["_score"], 3),
            "texto": h["_source"].get("texto", ""),
            "fonte": h["_source"].get("numero", h["_id"]),
            "tipo": h["_source"].get("tipo", "desconhecido")
        }
        for h in response["hits"]["hits"]
    ]

print("✅ Função de retrieval definida")

## 🔄 Passo 5 — Função de Reranking

In [ ]:
def rerank(query: str, docs: List[Dict], top_k: int = 5) -> List[Dict]:
    """
    Reordena lista de documentos usando BGE-Reranker cross-encoder.
    
    Processo:
    1. Cria pares (query, doc_texto) para cada documento
    2. Tokeniza os pares
    3. Executa forward pass no cross-encoder
    4. Aplica sigmoid para obter score 0-1
    5. Ordena por score decrescente
    6. Retorna top_k
    
    Args:
        query: Query do usuário
        docs: Lista de documentos do retrieval
        top_k: Número de documentos no resultado final
    
    Returns:
        Lista de documentos reordenados com 'score_reranker'
    """
    if not docs:
        return []
    
    if not RERANKER_OK:
        # Scores simulados para demonstração
        import random
        random.seed(42)
        scored = [{**d, "score_reranker": round(random.uniform(0.3, 0.98), 4)} for d in docs]
        return sorted(scored, key=lambda x: x["score_reranker"], reverse=True)[:top_k]
    
    texts = [d["texto"] for d in docs]
    pairs = [[query, t] for t in texts]
    
    all_scores = []
    BATCH_SIZE = 8
    
    for i in range(0, len(pairs), BATCH_SIZE):
        batch = pairs[i:i + BATCH_SIZE]
        inputs = reranker_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            logits = reranker_model(**inputs).logits.squeeze(-1)
            scores = torch.sigmoid(logits).cpu().numpy()
        
        all_scores.extend(scores.tolist() if scores.ndim > 0 else [float(scores)])
    
    scored = [{**d, "score_reranker": round(s, 4)} for d, s in zip(docs, all_scores)]
    return sorted(scored, key=lambda x: x["score_reranker"], reverse=True)[:top_k]

print("✅ Função de reranking definida")
print()
print("📖 Arquitetura do cross-encoder:")
print("   Input:  [CLS] query [SEP] documento [SEP]")
print("   Output: sigmoid(W · h_[CLS]) → score ∈ [0, 1]")

## 🧪 Passo 6 — Executar Pipeline nas 5 Queries

In [ ]:
# Carregar queries do baseline
try:
    with open("../datasets/corpus_juridico_aula3.json", encoding="utf-8") as f:
        dataset = json.load(f)
    queries = [(q["id"], q["query_original"]) for q in dataset["queries_baseline"][:5]]
except Exception:
    queries = [
        ("q_001", "Podem prender alguém sem mandado?"),
        ("q_002", "Policial pode ler mensagens do celular do suspeito?"),
        ("q_003", "O suspeito é obrigado a falar na delegacia?"),
        ("q_004", "O advogado pode ver o inquérito policial?"),
        ("q_005", "Como provar que o réu tinha intenção de matar?"),
    ]

resultados = []

for qid, query in queries:
    print(f"\n🔍 [{qid}] {query}")
    
    # RETRIEVAL
    t0 = time.time()
    docs_retrieval = retrieval_opensearch(query, top_k=20)
    t_retrieval = time.time() - t0
    
    # RERANKING  
    t0 = time.time()
    docs_reranked = rerank(query, docs_retrieval, top_k=5)
    t_reranking = time.time() - t0
    
    # Registrar mudanças de ranking
    rank_map_antes = {d["id"]: i+1 for i, d in enumerate(docs_retrieval[:5])}
    
    mudancas = []
    for new_rank, doc in enumerate(docs_reranked, 1):
        old_rank = rank_map_antes.get(doc["id"], ">5")
        mudancas.append({
            "rank_antes": old_rank,
            "rank_depois": new_rank,
            "score_bm25": doc.get("score_bm25", 0),
            "score_reranker": doc["score_reranker"],
            "fonte": doc["fonte"]
        })
    
    resultados.append({
        "qid": qid,
        "query": query,
        "mudancas": mudancas,
        "t_retrieval_ms": round(t_retrieval * 1000, 1),
        "t_reranking_ms": round(t_reranking * 1000, 1)
    })
    
    print(f"   Retrieval: {t_retrieval*1000:.0f}ms | Reranking: {t_reranking*1000:.0f}ms")

print("\n✅ Pipeline concluído para todas as queries!")

## 📊 Passo 7 — Tabela Comparativa

In [ ]:
print("📊 TABELA COMPARATIVA: Top-5 Antes vs Depois do Reranking")
print("=" * 75)

for r in resultados:
    print(f"\n━━ [{r['qid']}] {r['query'][:60]}...")
    print(f"{'#Antes':8} {'#Depois':8} {'BM25':8} {'Reranker':10} {'Fonte'}")
    print("-" * 60)
    for m in r["mudancas"]:
        seta = "↑" if str(m["rank_antes"]) > str(m["rank_depois"]) else ("↓" if str(m["rank_antes"]) < str(m["rank_depois"]) else "─")
        print(f"  {str(m['rank_antes']):6}  {m['rank_depois']:6}  {seta}  {m['score_bm25']:.3f}   {m['score_reranker']:.4f}    {m['fonte']}")
    print(f"  ⏱  Retrieval: {r['t_retrieval_ms']}ms | Reranking: {r['t_reranking_ms']}ms")

## 📈 Passo 8 — Análise de Impacto

In [ ]:
print("📈 ANÁLISE DE IMPACTO DO RERANKING")
print("=" * 55)
print()

total_mudancas = 0
maior_delta = 0
query_maior_delta = ""

for r in resultados:
    mudancas_nessa_query = 0
    for m in r["mudancas"]:
        antes = m["rank_antes"]
        depois = m["rank_depois"]
        if str(antes) != str(depois) and antes != ">5":
            mudancas_nessa_query += abs(int(antes) - int(depois))
    
    total_mudancas += mudancas_nessa_query
    if mudancas_nessa_query > maior_delta:
        maior_delta = mudancas_nessa_query
        query_maior_delta = r["query"]
    
    print(f"[{r['qid']}] Delta total de posições: {mudancas_nessa_query}")

print()
print(f"📌 Query com maior impacto do reranking:")
print(f"   '{query_maior_delta}'")
print(f"   Delta total: {maior_delta} posições")
print()

# Tempo médio
tempo_medio_reranking = sum(r["t_reranking_ms"] for r in resultados) / len(resultados)
tempo_medio_retrieval = sum(r["t_retrieval_ms"] for r in resultados) / len(resultados)
print(f"⏱  Tempo médio:")
print(f"   Retrieval: {tempo_medio_retrieval:.0f}ms")
print(f"   Reranking: {tempo_medio_reranking:.0f}ms")
print(f"   Overhead do reranking: +{tempo_medio_reranking/max(tempo_medio_retrieval,1):.1f}x")
print()
print("✅ LAB 2 CONCLUÍDO!")
print()
print("📋 CHECKLIST DE ENTREGA:")
print("   [ ] BGE-Reranker carregado e funcional")
print("   [ ] Pipeline retrieval + reranking executado para 5 queries")
print("   [ ] Tabela comparativa antes/depois gerada")
print("   [ ] Análise de impacto com identificação da query de maior mudança")
print("   [ ] Tempo de overhead do reranking registrado")